In [1]:
import sys
import os
from jiwer import wer
import evaluate
import re
from models.whisper_transcriber import WhisperTranscriber, DanishTranscriber
from models.translator import ChunkTranslator
from utils.audio_preprocessing import preprocess_audio
from utils.audio_streaming import stream_audio
from utils.num_to_words import numbers_to_words


# Get the absolute path to the project root (parent directory of the current folder)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to sys.path
sys.path.append(project_root)

c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\ctranslate2\__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1. Speech to text

In [2]:
transcriber = WhisperTranscriber()
# transcriber = DanishTranscriber()

translator = ChunkTranslator("Helsinki-NLP/opus-mt-en-da", context_length=2, num_beams=2)

data_path = "data/english/speaker_3_final.wav"
audio = preprocess_audio(data_path)

transcription_text = ""
translated_text = ""

for chunk in stream_audio(audio, frame_ms=200):
    transcription = transcriber.add_audio_chunk(chunk)
    if isinstance(transcription, str):
        transcription_text += " " + transcription
        translated = translator.translate_chunk(transcription)
        if isinstance(translated, str):
            translated_text += " " + translated


# At the end, flush any remaining audio
final_transcription = transcriber.flush()
transcription_text += " " + final_transcription
if isinstance(final_transcription, str):
    final_translation = translator.translate_chunk(final_transcription)
    if isinstance(final_translation, str):
        translated_text += " " + final_translation

translator.reset_context()

Loading Silero-VAD …
Silero-VAD initialised!
Loading Faster Whisper model: base.en …
Whisper model loaded!


c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Transcribed text: So I've known a lot of fish in my life.
Transcribed text: I've loved only two. That first one was...
Transcribed text: It was more like a passionate affair. It was a beautiful...
Transcribed text: fish. Flavorful, textured, meaty, a best-seller...
Transcribed text: on the menu. What a fish. Even better.
Transcribed text: It was farm raised to the supposed at highest standards of sustainable.
Transcribed text: So you can feel good about selling it. I was in a relationship with this beauty.
Transcribed text: For several months. One day the head of the company called and asked me to make a decision.
Transcribed text: He asked if I'd speak at an event about the farm sustainability. Absolutely I said. Here is the company.
Transcribed text: What's become this unimaginable problem for us chefs?
Transcribed text: How do we keep fish on our menus? For the past 50 years...
Transcribed text: We've been fishing the seas like we clear-cut forests. It's hard...
Transcribed text: to

In [ ]:
def clean_text(text):
    text = re.sub(r"-", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.lower()
    return text

transcription_text = clean_text(transcription_text)
translated_text = clean_text(translated_text)

In [11]:
transcription_text = numbers_to_words(transcription_text, lang='en')
translated_text = numbers_to_words(translated_text, lang='da')

In [12]:
with open("data/english/speaker_3_final.txt", "r", encoding="utf-8") as f:
    ref_transcription = f.read()
ref_transcription = clean_text(ref_transcription)
ref_transcription = numbers_to_words(ref_transcription, lang='en')

wer_score = wer(reference=ref_transcription, hypothesis=transcription_text)
wer_score

0.134185303514377

In [13]:
ref_transcription, transcription_text

('so ive known a lot of fish in my life ive loved only two that first one was a it was more like a passionate affair it was a beautiful fish flavorful textured meaty a best seller on the menu what a fish even better it was farm raised to the supposed highest standards of sustainability so you could feel good about selling it i was in a relationship with this beauty for several months one day the head of the company called and asked if id speak at an event about the farms sustainability absolutely i said here was a company trying to solve what s become this unimaginable problem for our chefs how do we keep fish on our menus for the past fifty years weve been fishing the seas like we clear cut forests its hard to overstate the destruction ninety percent of large fish the ones we love the tunas the halibuts the salmons swordfish theyve collapsed there was nothing left so for better or for worse aquaculture fish farming is going to be a part of our future a lot of arguments against it fish

### Text translation

In [15]:
with open("data/speaker_2.txt", "r", encoding="utf-8") as f:
    ref_translation = f.read()

In [ ]:
comet = evaluate.load("comet")
comet_score = comet.compute(
    predictions=[translated_text],
    references=[ref_translation],
    sources=[transcription_text],
)

comet_score

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 3545.48it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\aneas\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\pytorch_lightning\core\saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


{'mean_score': 0.992570161819458, 'scores': [0.992570161819458]}

In [ ]:
# translated_text, ref_translation, transcription_text

### Clean danish files

In [5]:
dk_txt_list = []
for i in range(1, 11):
    filename = f"data/danish/dk_speaker_{i}.txt"
    with open(filename, "r", encoding="utf-8") as f:
        dk_txt = f.read()
        dk_txt_list.append(dk_txt)

In [6]:
import os
import re
from utils.num_to_words import numbers_to_words

os.makedirs("data/danish", exist_ok=True)

for i, dk_txt in enumerate(dk_txt_list, start=1):
    cleaned = clean_text(dk_txt)
    converted = numbers_to_words(cleaned, lang='da', split_abbreviations=False)
    out_filename = f"data/danish/dk_speaker_{i}_final.txt"
    with open(out_filename, "w", encoding="utf-8") as f:
        f.write(converted)